# 1.Human In The Loop

In [ ]:
from json import tool
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command


# 声明状态
class State(TypedDict):
    username: str


# 声明节点
def node_a(state: State) -> dict:
    username = interrupt("请输入你的名字")
    return {
        "username": username,
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("node_a", node_a)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)


In [ ]:
# 恢复执行
msg = res['__interrupt__'][0].value
username = input(msg)

resume_res = graph.invoke(Command(resume=username), config=config)
print(resume_res)

# 2. 多个并行中断

In [ ]:

from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command


# 声明状态
class State(TypedDict):
    username: str
    age: str


# 声明节点
def node_a(state: State) -> dict:
    username = interrupt("请输入你的名字")
    return {
        "username": username,
    }


def node_b(state: State) -> dict:
    age = interrupt("请输入你的年龄")
    return {
        "age": age,
    }


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)

# 执行
config = {"configurable": {"thread_id": "123"}}
res = graph.invoke({}, config=config)
print(res)


In [ ]:
# 恢复执行
resume_map = {}  # { id : msg }
for i in res['__interrupt__']:
    ask_msg = input(f"{i.value}")
    resume_map[i.id] = ask_msg
resume_res = graph.invoke(Command(resume=resume_map), config=config)
print(resume_res)

# 3.审批模式

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from typing import TypedDict, Literal

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import Command
from langgraph.types import interrupt
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model("deepseek-flash", )


# res = model.invoke([HumanMessage("你好"),])
# rprint(res)


# 声明状态
class State(TypedDict):
    topic: str
    poem: str
    is_approved: bool


# 声明节点
def approved_node(state: State) -> Command[Literal["llm_node", "default_node"]]:
    is_approved = interrupt("是否同意调用模型？")
    goto = "llm_node" if is_approved else "default_node"
    return Command(goto=goto, update={"is_approved": is_approved})


def llm_node(state: State) -> dict:
    topic = state["topic"]
    poem = model.invoke([HumanMessage(f"写一首关于{topic}的短诗")]).content
    return {"poem": poem}


def default_node(state: State) -> dict:
    return {"poem": "请求被拒绝"}


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("approved_node", approved_node)
builder.add_node("llm_node", llm_node)
builder.add_node("default_node", default_node)
builder.add_edge(START, "approved_node")
builder.add_edge("llm_node", END)
builder.add_edge("default_node", END)

# 创建检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)
config = {"configurable": {"thread_id": "1"}}

# 执行 -> 触发中断
res = graph.invoke({"topic": "月亮"}, config=config)
print(res)

In [ ]:
# 用户审批 -> 恢复执行
ask = res["__interrupt__"][0].value
approve_res = input(ask).strip().lower() in ("y", "yes", "是", "1")

resume_res = graph.invoke(Command(resume=approve_res), config=config)
print(resume_res)


# 4.审核与编辑模式

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from typing import Literal
from rich import print as rprint
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.types import Command, interrupt
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model("deepseek-flash")


# 声明状态：MessagesState 自带 messages: Annotated[list[AnyMessage], add_messages]
# 用户输入/审核意见(HumanMessage)、模型稿件(AIMessage)统一追加进 messages，天然就是完整对话记录
class State(MessagesState):
    pass


# 声明节点
def llm_node(state: State) -> dict:
    resp = model.invoke(state["messages"])
    return {"messages": [resp]}


def review_node(state: State) -> Command[Literal["llm_node", END]]:
    review_msg = interrupt(
        {"instruction": "请审核大模型生成的内容，输入 y/yes 通过；或直接输入修改意见", }
    )
    if str(review_msg).strip().lower() in ("y", "yes"):
        return Command(goto=END)
    # 审核意见作为新的 HumanMessage 追加，goto 回 llm_node 按意见修改
    return Command(goto="llm_node", update={"messages": [HumanMessage(str(review_msg))]})


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("review_node", review_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "review_node")
builder.add_edge("review_node", END)

# 创建检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)
config = {"configurable": {"thread_id": "1"}}

# 执行 -> 触发中断：初始输入作为第一条消息
res = graph.invoke({"messages": [HumanMessage("写一首关于月亮的短诗，只需要提供短诗")]}, config=config)

# 人工审核循环：输入修改意见 -> 督促大模型按意见优化；输入 yes -> 通过结束
while res.get("__interrupt__"):
    user_in = input(res["__interrupt__"][0].value['instruction'])
    res = graph.invoke(Command(resume=user_in), config=config)

rprint(res)

In [ ]:
# 打印图结构
from IPython.display import display
display(graph)

# 5.工具审批模式

In [ ]:
from typing import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState
from langgraph.types import Command, interrupt

load_dotenv(override=True)

model = init_chat_model("deepseek-flash")


# 两个工具：让模型一轮发起多个 tool_call，才能看出逐个审批的效果
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = {t.name: t for t in [get_weather, get_news]}
model_with_tools = model.bind_tools(tools.values())


# 声明状态
class State(MessagesState):
    decisions: dict  # {tool_call_id: 是否同意}


# 声明节点
def llm_node(state: State) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


def review_node(state: State) -> Command[Literal["tool_node", "llm_node", END]]:
    tool_calls = state["messages"][-1].tool_calls
    if not tool_calls:  # 模型直接回答，没有工具调用
        return Command(goto=END)
    # 工具执行前中断，列出全部待审批调用；恢复值是 {tool_call_id: 是否同意}
    decisions = interrupt([
        {
            "id": tc["id"],
            "name": tc["name"],
            "args": tc["args"]
        }
        for tc in tool_calls
    ])
    if any(decisions.values()):
        return Command(goto="tool_node", update={"decisions": decisions})
    # 全部拒绝 -> 回填拒绝 ToolMessage，回到 llm_node 让模型直接回答
    return Command(goto="llm_node", update={"messages": [
        ToolMessage(content=f"用户拒绝执行 {tc['name']}，请勿重试，直接回答", tool_call_id=tc["id"])
        for tc in tool_calls
    ]})


def tool_node(state: State) -> dict:
    last = state["messages"][-1]
    decisions = state["decisions"]
    # 同意的调用真正执行；拒绝的回填拒绝 ToolMessage，保证每个 tool_call_id 都有应答
    messages = []
    for tc in last.tool_calls:
        if decisions.get(tc["id"]):
            result = str(tools[tc["name"]].invoke(tc["args"]))
        else:
            result = f"用户拒绝执行 {tc['name']}，请勿重试，直接回答"
        messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    return {"messages": messages}


# 构建图结构
builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("review_node", review_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "review_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("review_node", END)

# 使用中断必须设置检查点
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer)
config = {"configurable": {"thread_id": "1"}}

# 执行 -> 触发中断：等待用户对每个 tool_call 逐个审批
# 若模型不支持并行工具调用而分多轮发起，每轮各触发一次中断，恢复循环同样处理
res = graph.invoke({"messages": [HumanMessage("查一下北京的天气，再查一下科技新闻")]}, config=config)

In [ ]:
# 逐个审批 -> 恢复执行：对中断里的每个调用输入 y/n，按 tool_call_id 组装决策字典
while res.get("__interrupt__"):
    decisions = {}
    for call in res["__interrupt__"][0].value:
        user_in = input(f"是否允许 {call['name']}({call['args']})? (y/n)")
        decisions[call["id"]] = user_in.strip().lower() in ("y", "yes", "是", "1")
    res = graph.invoke(Command(resume=decisions), config=config)

rprint(res)


In [ ]:
# 打印图结构
from IPython.display import display
display(graph)